In [ ]:
# 导入环境变量
from dotenv import load_dotenv
import os

load_dotenv(override=True)
BASE_URL = os.getenv("MINIMAX_BASE_URL")
API_KEY = os.getenv("MINIMAX_API_KEY")
MODEL_ID = os.getenv("MINIMAX_MODEL_ID")

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="anthropic",  # Mini-Max模型通过anthropic的模型规范进行交互
    model=MODEL_ID,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# 工具的定义
所有工具均继承自 BaseTool 抽象基类
每个工具都包含三个关键组成部分：
- 名称：工具的唯一标识符，让模型能够在众多工具中准确识别它
- 描述：一段清晰的自然语言说明，告知模型工具功能和使用场景
- 执行逻辑：工具的实际执行代码

In [1]:
from langchain.tools import tool

@tool
def multiplay(a: int, b: int) -> int:
    """返回两个整数相乘的结果"""
    return a * b

# 工具属性
print("工具属性：")
print(multiplay.name)
print(multiplay.description)
print(multiplay.args)

# 手动调用
print(f'手动调用结果：{multiplay.invoke({"a": 3, "b": 4})}')

工具属性：
multiplay
返回两个整数相乘的结果
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
手动调用结果：12


# 工具的基本使用

In [2]:
from langchain.tools import tool

# 对于多参数需求的工具，需要传入一个字典对象
@tool
def process_user_data(user_id: str, is_active: bool) -> str:
    """
    根据用户ID和活动状态，更新用户记录

    参数：
      user_id (str): 用户的唯一标识符
      is_active (bool): 表示用户是否处于活动状态 (True / False)
    """
    status = "激活" if is_active else "禁用"
    return f"用户 ID: {user_id} 已成功更新为 {status} 状态"

print("调用工具 process_user_data")
print(f'执行结果：{process_user_data.invoke({"user_id": "U001", "is_active": True})}')

调用工具 process_user_data
执行结果：用户 ID: U001 已成功更新为 激活 状态


In [3]:
# 对于只需要一个参数的工具，可以直接传对应的值
@tool
def simple_greeting(name: str) -> str:
    """对给定的人名说一句问候语"""
    return f"你好，{name}！"

print("调用工具 simple_greeting")
print(f'执行结果：{simple_greeting.invoke("护道者")}')

调用工具 simple_greeting
执行结果：你好，护道者！


In [4]:
# 通过 @tool 直接定义工具属性

@tool(
    "exchange_rate_helper",
    description="用于进行人民币与美元之间的汇率换算。当用户询问金额转换时使用此工具。"
)
def convert_currency(amount: float, to_currency: str) -> str:
    """执行基础的货币汇率换算（示例比例：1USD = 7.2CNY）"""
    rate = 7.2
    if to_currency.lower() == "usd":
        result = amount / rate
        return f"{amount} 人民币 ≈ {result:.2f} 美元"
    elif to_currency.lower() == "cny":
        result = amount * rate
        return f"{amount} 美元 约等于 {result:.2f} 人民币"
    else:
        return "暂不支持该货币类型"

# 查看工具信息
print(convert_currency.name)
print(convert_currency.description)
# 调用工具
print(convert_currency.invoke({"amount": 100, "to_currency": "USD"}))

exchange_rate_helper
用于进行人民币与美元之间的汇率换算。当用户询问金额转换时使用此工具。
100.0 人民币 ≈ 13.89 美元
